<a href="https://colab.research.google.com/github/Rini43/NLP_Project_Receipe_Reviews_and_Feedback/blob/main/NLP_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Libraries

In [ ]:
# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text Processing
import string
import re
import html

# PyTorch (Required for BERT)
# import torch

# NLTK Libraries
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

# NLTK Preprocessing Tools
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Train-Test Split
from sklearn.model_selection import train_test_split

# Text Vectorization (Bag of Words & TF-IDF)
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Classical Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Model Evaluation Metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report)

# BERT Tokenizer and Pre-trained Model
# from transformers import AutoTokenizer
# from transformers import AutoModel

# Deep learning models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

import joblib

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


# Read the Data

In [ ]:
# filepath of the file
# Load the dataset from the cloned GitHub repository

filepath = "https://raw.githubusercontent.com/Rini43/NLP_Project_Receipe_Reviews_and_Feedback/main/Recipe%20Reviews%20and%20User%20Feedback%20Dataset.csv"
df_review = pd.read_csv(filepath)
df_review.head(5)

,Unnamed: 0,recipe_number,recipe_code,recipe_name,comment_id,user_id,user_name,user_reputation,created_at,reply_count,thumbs_up,thumbs_down,stars,best_score,text
0,0,1,14299,Creamy White Chili,sp_aUSaElGf_14299_c_2G3aneMRgRMZwXqIHmSdXSG1hEM,u_9iFLIhMa8QaG,Jeri326,1,1665619889,0,0,0,5,527,"I tweaked it a little, removed onions because ..."
1,1,1,14299,Creamy White Chili,sp_aUSaElGf_14299_c_2FsPC83HtzCsQAtOxlbL6RcaPbY,u_Lu6p25tmE77j,Mark467,50,1665277687,0,7,0,5,724,Bush used to have a white chili bean and it ma...
2,2,1,14299,Creamy White Chili,sp_aUSaElGf_14299_c_2FPrSGyTv7PQkZq37j92r9mYGkP,u_s0LwgpZ8Jsqq,Barbara566,10,1664404557,0,3,0,5,710,I have a very complicated white chicken chili ...
3,3,1,14299,Creamy White Chili,sp_aUSaElGf_14299_c_2DzdSIgV9qNiuBaLoZ7JQaartoC,u_fqrybAdYjgjG,jeansch123,1,1661787808,2,2,0,0,581,"In your introduction, you mentioned cream chee..."
4,4,1,14299,Creamy White Chili,sp_aUSaElGf_14299_c_2DtZJuRQYeTFwXBoZRfRhBPEXjI,u_XXWKwVhKZD69,camper77,10,1664913823,1,7,0,0,820,Wonderful! I made this for a &#34;Chili/Stew&#...


# EDA

In [ ]:
# Identifying numerical and categorical data

df_review.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18182 entries, 0 to 18181
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Unnamed: 0       18182 non-null  int64 
 1   recipe_number    18182 non-null  int64 
 2   recipe_code      18182 non-null  int64 
 3   recipe_name      18182 non-null  object
 4   comment_id       18182 non-null  object
 5   user_id          18182 non-null  object
 6   user_name        18182 non-null  object
 7   user_reputation  18182 non-null  int64 
 8   created_at       18182 non-null  int64 
 9   reply_count      18182 non-null  int64 
 10  thumbs_up        18182 non-null  int64 
 11  thumbs_down      18182 non-null  int64 
 12  stars            18182 non-null  int64 
 13  best_score       18182 non-null  int64 
 14  text             18180 non-null  object
dtypes: int64(10), object(5)
memory usage: 2.1+ MB


In [ ]:
# Identifying dataset size

df_review.shape

(18182, 15)

In [ ]:
# Statistical Summary

df_review.describe()

,Unnamed: 0,recipe_number,recipe_code,user_reputation,created_at,reply_count,thumbs_up,thumbs_down,stars,best_score
count,18182.000000,18182.000000,18182.000000,18182.000000,1.818200e+04,18182.000000,18182.000000,18182.000000,18182.000000,18182.000000
mean,121.465295,38.689363,21773.667253,2.159608,1.623710e+09,0.014630,1.089264,0.549335,4.288802,153.162138
std,116.747893,29.786647,23965.109637,10.014666,5.468697e+06,0.137974,4.201004,3.470124,1.544786,141.075316
min,0.000000,1.000000,386.000000,0.000000,1.613035e+09,0.000000,0.000000,0.000000,0.000000,0.000000
25%,45.000000,12.000000,6086.000000,1.000000,1.622717e+09,0.000000,0.000000,0.000000,5.000000,100.000000
50%,91.000000,33.000000,14600.000000,1.000000,1.622718e+09,0.000000,0.000000,0.000000,5.000000,100.000000
75%,150.000000,64.000000,33121.000000,1.000000,1.622718e+09,0.000000,0.000000,0.000000,5.000000,100.000000
max,724.000000,100.000000,191775.000000,520.000000,1.665756e+09,3.000000,106.000000,126.000000,5.000000,946.000000


In [ ]:
# Target Variable Analysis

df_review["stars"].value_counts()

# 1–5 are actual rating classes.
# 0 is not a rating, it means the user did not provide a rating.

,count
stars,
5,13829
0,1696
4,1655
3,490
1,280
2,232


In [ ]:
# Trying to findout if there is any missing value present

df_review.isna().sum()

,0
Unnamed: 0,0
recipe_number,0
recipe_code,0
recipe_name,0
comment_id,0
user_id,0
user_name,0
user_reputation,0
created_at,0
reply_count,0


In [ ]:
# Handling the missing value

df_review.dropna(subset=["text"], inplace=True)

In [ ]:
# After handling the misssing values

df_review.isna().sum()

,0
Unnamed: 0,0
recipe_number,0
recipe_code,0
recipe_name,0
comment_id,0
user_id,0
user_name,0
user_reputation,0
created_at,0
reply_count,0


In [ ]:
# Duplicate Rows

df_review.duplicated().sum()

np.int64(0)

In [ ]:
# Review Length Analysis

df_review["review_length"] = df_review["text"].apply(len)
df_review[["text", "review_length"]].head(5)

,text,review_length
0,"I tweaked it a little, removed onions because ...",355
1,Bush used to have a white chili bean and it ma...,138
2,I have a very complicated white chicken chili ...,354
3,"In your introduction, you mentioned cream chee...",93
4,Wonderful! I made this for a &#34;Chili/Stew&#...,238


In [ ]:
# removing unwanted colums
df_review.drop(columns = ['Unnamed: 0','recipe_number','recipe_code','comment_id','user_id','user_reputation','created_at','reply_count','best_score'],inplace = True)

In [ ]:
df_review.drop(columns = ['thumbs_up','thumbs_down','user_name'],inplace = True)

# NLP Preprocessing

## Lowercasing

In [ ]:
# Convert to lowercase

df_review['recipe_name'] = df_review['recipe_name'].str.lower()
df_review['text'] = df_review['text'].astype(str).str.lower()

## Punctuation

In [ ]:
# Remove punctuation

text_columns = ["recipe_name", "text"]

for col in text_columns:
    df_review[col] = df_review[col].fillna("").str.lower()
    df_review[col] = df_review[col].str.translate(
        str.maketrans("", "", string.punctuation))

## URL

In [ ]:
# Checking for the URLs

url_count = df_review["text"].str.contains(r"http\S+|www\.\S+", regex=True, na=False).sum()
print(f"Reviews containing URLs in text: {url_count}")

url_count = df_review["recipe_name"].str.contains(r"http\S+|www\.\S+", regex=True, na=False).sum()
print(f"Reviews containing URLs in recipe name: {url_count}")

Reviews containing URLs in text: 34
Reviews containing URLs in recipe name: 0


In [ ]:
# Remove URLs

df_review["text"] = df_review["text"].str.replace(r"http\S+|www\.\S+", "", regex=True)

## HTML

In [ ]:
# Convert HTML entities to normal characters and ensure text is in string format

df_review["text"] = df_review["text"].apply(
    lambda x: html.unescape(str(x))
)

## Emoji

In [ ]:
# Checking for emojis

# Compile emoji pattern once

emoji_pattern = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "]+",
    flags=re.UNICODE)

# Check multiple columns
text_columns = ["text", "recipe_name"]

for col in text_columns:
    emoji_count = df_review[col].apply(
        lambda x: bool(emoji_pattern.search(str(x)))).sum()

    print(f"Rows containing emojis in '{col}': {emoji_count}")

Rows containing emojis in 'text': 11
Rows containing emojis in 'recipe_name': 0


In [ ]:
# Removing the emojis

df_review["text"] = df_review["text"].apply(
    lambda x: emoji_pattern.sub("", str(x)))

## Numbers

In [ ]:
# Checking for numbers

text_columns = ["text", "recipe_name"]

for col in text_columns:
    number_count = df_review[col].str.contains(r"\d", regex=True, na=False).sum()
    print(f"Rows containing numbers in '{col}': {number_count}")

Rows containing numbers in 'text': 8923
Rows containing numbers in 'recipe_name': 0


In [ ]:
# Display rows containing numbers

number_rows = df_review[df_review["text"].str.contains(r"\d", regex=True, na=False)]
number_rows["text"].head(20)

,text
4,wonderful i made this for a 34chilistew34 nigh...
6,wow this recipe is excellent as written the ...
7,this is delicious and i make it often one such...
8,i absolutely love this recipe i39ve tweaked it...
11,best white chili recipe i39ve had i served it ...
12,this recipe was excellent i added the cream ch...
14,fantastic but mild i added half a carolina rea...
21,this is our goto chicken chili recipe and has ...
23,this is just white chicken chili with i first ...
24,wow total wow totally delicious 5 stars plus


In [ ]:
# Convert the 'text' column to string data type
df_review["text"] = df_review["text"].astype("string")

# Check the data type of the 'text' column
print(df_review["text"].dtype)

string


## Tokenization

In [ ]:
text = df_review['text'].apply(word_tokenize)

## Stopword Removal

In [ ]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
  tokens = word_tokenize(text)
  filtered = [word for word in tokens if word not in stop_words]
  return " ".join(filtered)

df_review['text'] = df_review['text'].apply(remove_stopwords)

## Lemmatization

In [ ]:
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
  tokens = word_tokenize(text)
  lemmas = [lemmatizer.lemmatize(word) for word in tokens]
  return " ".join(lemmas)

df_review['text'] = df_review['text'].apply(lemmatize_text)

# Classical Machine Learning Models

In [ ]:
# Define Features and Target

X = df_review["text"]

# Target variable
y = df_review["stars"]

### Train-Test Split

In [ ]:
# Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=y)

# The vectorizer is fitted only on the training data to prevent data leakage.
# The learned vocabulary is then used to transform both the training and test data.

In [ ]:
# Verify the shapes

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (14544,)
X_test : (3636,)
y_train: (14544,)
y_test : (3636,)


# Text Vectorization

## Bag of Words

In [ ]:
# Bag of Words

bow = CountVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

# we are checking for the word counts

In [ ]:
# Checking the Output

print("Training Shape :", X_train_bow.shape)
print("Testing Shape  :", X_test_bow.shape)

print("Vocabulary Size :", len(bow.vocabulary_))

Training Shape : (14544, 5000)
Testing Shape  : (3636, 5000)
Vocabulary Size : 5000


In [ ]:
# View Vocabulary

feature_names = bow.get_feature_names_out()
print(feature_names[:50])

['10' '10 min' '10 minute' '10 star' '10 year' '100' '11' '112' '112 cup'
 '12' '12 cup' '12 hour' '12 lb' '12 minute' '12 oz' '12 pound'
 '12 recipe' '12 teaspoon' '12 tsp' '12 year' '13' '13 cup' '13 pan'
 '13x9' '14' '14 cup' '14 teaspoon' '14 tsp' '15' '15 cup' '15 min'
 '15 minute' '15 oz' '15 year' '15x10' '16' '16 oz' '18' '18 cup' '1996'
 '1lb' '1st' '1st time' '1tsp' '20' '20 min' '20 minute' '20 year' '23'
 '23 cup']


In [ ]:
# Evaluation Function

def evaluate_model(model, X_test, y_test):

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(y_test, y_pred,
        average="weighted")

    recall = recall_score(y_test, y_pred,
        average="weighted")

    f1 = f1_score(y_test, y_pred,
        average="weighted")

    print("Accuracy :", round(accuracy,4))
    print("Precision:", round(precision,4))
    print("Recall   :", round(recall,4))
    print("F1 Score :", round(f1,4))

    print("\nClassification Report\n")
    print(classification_report(y_test,y_pred))

    print("\nConfusion Matrix\n")
    print(confusion_matrix(y_test,y_pred))

    return accuracy, precision, recall, f1

In [ ]:
# Creating Results List

results = []

In [ ]:
# Train Logistic Regression

lr = LogisticRegression(
    max_iter=1000,
    random_state=42)

lr.fit(X_train_bow, y_train)

accuracy, precision, recall, f1 = evaluate_model(
    lr, X_test_bow, y_test)

results.append({
    "Embedding":"Bag of Words",
    "Model":"Logistic Regression",
    "Accuracy":accuracy,
    "Precision":precision,
    "Recall":recall,
    "F1 Score":f1})

Accuracy : 0.7478
Precision: 0.7012
Recall   : 0.7478
F1 Score : 0.7191

Classification Report

              precision    recall  f1-score   support

           0       0.30      0.18      0.23       339
           1       0.32      0.25      0.28        56
           2       0.25      0.06      0.10        47
           3       0.27      0.17      0.21        98
           4       0.32      0.22      0.26       331
           5       0.83      0.92      0.87      2765

    accuracy                           0.75      3636
   macro avg       0.38      0.30      0.33      3636
weighted avg       0.70      0.75      0.72      3636


Confusion Matrix

[[  61    7    1   11   25  234]
 [  11   14    4    3    3   21]
 [   2    7    3    9    9   17]
 [   5    8    1   17   19   48]
 [  24    3    2   10   74  218]
 [  97    5    1   12  100 2550]]


In [ ]:
# Train SVM

svm = SVC(
    kernel="linear",
    probability=True,
    random_state=42)

svm.fit(X_train_bow, y_train)

accuracy, precision, recall, f1 = evaluate_model(
    svm, X_test_bow, y_test)

results.append({
    "Embedding":"Bag of Words",
    "Model":"SVM",
    "Accuracy":accuracy,
    "Precision":precision,
    "Recall":recall,
    "F1 Score":f1})

Accuracy : 0.7085
Precision: 0.6946
Recall   : 0.7085
F1 Score : 0.7008

Classification Report

              precision    recall  f1-score   support

           0       0.25      0.24      0.25       339
           1       0.25      0.27      0.26        56
           2       0.07      0.06      0.07        47
           3       0.19      0.20      0.20        98
           4       0.30      0.22      0.25       331
           5       0.83      0.86      0.85      2765

    accuracy                           0.71      3636
   macro avg       0.32      0.31      0.31      3636
weighted avg       0.69      0.71      0.70      3636


Confusion Matrix

[[  82    8    5   12   19  213]
 [  12   15    7    5    3   14]
 [   2   10    3   11    7   14]
 [  12   10    8   20   13   35]
 [  30    4    5   21   73  198]
 [ 189   14   12   36  131 2383]]


In [ ]:
# Display Final Results

results_df = pd.DataFrame(results)
results_df

,Embedding,Model,Accuracy,Precision,Recall,F1 Score
0,Bag of Words,Logistic Regression,0.747800,0.701211,0.747800,0.719091
1,Bag of Words,SVM,0.708471,0.694571,0.708471,0.700810


## TF-IDF Vectorization

In [ ]:
# TF-IDF

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,3),
    min_df=2,
    max_df=0.95)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# we are checking for the weighted word frequencies

In [ ]:
# Checking the Output

print("Training Shape :", X_train_tfidf.shape)
print("Testing Shape  :", X_test_tfidf.shape)

print("Vocabulary Size :", len(tfidf.vocabulary_))

Training Shape : (14544, 10000)
Testing Shape  : (3636, 10000)
Vocabulary Size : 10000


In [ ]:
# Train Logistic Regression

lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'
)

lr.fit(X_train_tfidf, y_train)

accuracy, precision, recall, f1 = evaluate_model(
    lr, X_test_tfidf, y_test)

results.append({
    "Embedding":"TF-IDF",
    "Model":"Logistic Regression",
    "Accuracy":accuracy,
    "Precision":precision,
    "Recall":recall,
    "F1 Score":f1})

Accuracy : 0.6218
Precision: 0.7368
Recall   : 0.6218
F1 Score : 0.6636

Classification Report

              precision    recall  f1-score   support

           0       0.26      0.41      0.31       339
           1       0.29      0.46      0.35        56
           2       0.12      0.15      0.13        47
           3       0.17      0.31      0.22        98
           4       0.24      0.45      0.31       331
           5       0.90      0.69      0.78      2765

    accuracy                           0.62      3636
   macro avg       0.33      0.41      0.35      3636
weighted avg       0.74      0.62      0.66      3636


Confusion Matrix

[[ 139   14   12   25   41  108]
 [  10   26    6    8    3    3]
 [   1   12    7   12   11    4]
 [   9   12    7   30   25   15]
 [  45    3    9   31  150   93]
 [ 341   24   18   75  398 1909]]


In [ ]:
# Train SVM

svm = SVC(
    kernel="linear",
    probability=True,
    random_state=42)

svm.fit(X_train_tfidf, y_train)

accuracy, precision, recall, f1 = evaluate_model(
    svm, X_test_tfidf, y_test)

results.append({
    "Embedding":"TF-IDF",
    "Model":"SVM",
    "Accuracy":accuracy,
    "Precision":precision,
    "Recall":recall,
    "F1 Score":f1})

Accuracy : 0.7786
Precision: 0.7191
Recall   : 0.7786
F1 Score : 0.7064

Classification Report

              precision    recall  f1-score   support

           0       0.48      0.09      0.14       339
           1       0.57      0.23      0.33        56
           2       0.00      0.00      0.00        47
           3       0.50      0.12      0.20        98
           4       0.57      0.08      0.14       331
           5       0.79      0.99      0.88      2765

    accuracy                           0.78      3636
   macro avg       0.48      0.25      0.28      3636
weighted avg       0.72      0.78      0.71      3636


Confusion Matrix

[[  29    5    1    1    1  302]
 [  10   13    1    1    3   28]
 [   1    2    0    5    3   36]
 [   3    3    0   12   10   70]
 [   7    0    0    5   26  293]
 [  11    0    0    0    3 2751]]


In [ ]:
# Display Final Results

results_df = pd.DataFrame(results)
results_df

,Embedding,Model,Accuracy,Precision,Recall,F1 Score
0,Bag of Words,Logistic Regression,0.747800,0.701211,0.747800,0.719091
1,Bag of Words,SVM,0.708471,0.694571,0.708471,0.700810
2,TF-IDF,Logistic Regression,0.621837,0.736835,0.621837,0.663644
3,TF-IDF,SVM,0.778603,0.719110,0.778603,0.706423


## BERT Embedding

* Dense contextual embeddings
* It takes much longer to generate embeddings.
* It uses more RAM and computation.

Therefore, in this project, only the BERT model implementation is provided as a reference and is not used for the final model comparison.

In [ ]:
# Loading the Pre-trained BERT

# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# bert_model = AutoModel.from_pretrained("bert-base-uncased")

In [ ]:
# BERT Embedding Function

# def get_bert_embeddings(texts):

#    embeddings = []
#    bert_model.eval()
#    with torch.no_grad():

#       for text in texts:

#            encoded = tokenizer(
#                str(text),
#                padding="max_length",
#                truncation=True,
#                max_length=128,
#                return_tensors="pt")

#            output = bert_model(**encoded)

#            cls_embedding = output.last_hidden_state[:,0,:].squeeze().numpy()
#            embeddings.append(cls_embedding)

#    return np.array(embeddings)

In [ ]:
# Generate Embeddings

# X_train_bert = get_bert_embeddings(X_train)
# X_test_bert = get_bert_embeddings(X_test)

In [ ]:
# Checking the Shape

# print(X_train_bert.shape)
# print(X_test_bert.shape)

In [ ]:
# Train Logistic Regression

# lr = LogisticRegression(
#    max_iter=1000,
#    random_state=42)

# lr.fit(X_train_bert, y_train)

# accuracy, precision, recall, f1 = evaluate_model(
#    lr, X_test_bert, y_test)

# results.append({
#    "Embedding":"BERT",
#    "Model":"Logistic Regression",
#   "Accuracy":accuracy,
#    "Precision":precision,
#    "Recall":recall,
#   "F1 Score":f1})

In [ ]:
# Train SVM

# svm = SVC(
#    kernel="linear",
#    probability=True,
#    random_state=42)

# svm.fit(X_train_bert, y_train)

# accuracy, precision, recall, f1 = evaluate_model(
#    svm, X_test_bert, y_test)

# results.append({
#    "Embedding":"BERT",
#    "Model":"SVM",
#    "Accuracy":accuracy,
#    "Precision":precision,
#    "Recall":recall,
#    "F1 Score":f1})

In [ ]:
# Display Final Results

# results_df = pd.DataFrame(results)
# results_df

# Deep Learning Model

### Tokenization

In [ ]:
# Step 1: Convert text into numerical sequences


# Maximum number of words to keep in the vocabulary
MAX_WORDS = 10000

# Maximum length of each sequence
MAX_LENGTH = 200

# Create tokenizer
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")

# Fit tokenizer on training data
tokenizer.fit_on_texts(X_train)

# Convert text into sequences
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Pad sequences so all have the same length
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LENGTH,
                            padding='post', truncating='post')

X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LENGTH,
                           padding='post', truncating='post')

### Building BiLSTM Model

In [ ]:
# Create Sequential model

model = Sequential()

# Word Embedding Layer
# Converts words into dense vector representations

model.add(Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LENGTH))

# Bidirectional LSTM Layer
# Reads the text in both forward and backward directions

model.add(Bidirectional(LSTM(64)))

# Dropout Layer
# Reduces overfitting

model.add(Dropout(0.5))

# Hidden Dense Layer

model.add(Dense(64, activation='relu'))

# Dropout

model.add(Dropout(0.3))

# Output Layer
# Number of neurons = Number of classes (5)

model.add(Dense(6, activation='softmax'))     # Multiclass

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
# Compile the model

model.compile(optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])

# Display model summary

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### Training the Model

In [ ]:
# Early stopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True)

# Train the model

history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop])

Epoch 1/5
364/364 ━━━━━━━━━━━━━━━━━━━━ 16s 22ms/step - accuracy: 0.7591 - loss: 0.8402 - val_accuracy: 0.7521 - val_loss: 0.7534
Epoch 2/5
364/364 ━━━━━━━━━━━━━━━━━━━━ 7s 20ms/step - accuracy: 0.7784 - loss: 0.6755 - val_accuracy: 0.7576 - val_loss: 0.7441
Epoch 3/5
364/364 ━━━━━━━━━━━━━━━━━━━━ 8s 21ms/step - accuracy: 0.8078 - loss: 0.5736 - val_accuracy: 0.7556 - val_loss: 0.8068
Epoch 4/5
364/364 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.8375 - loss: 0.4892 - val_accuracy: 0.7425 - val_loss: 0.8653
Epoch 5/5
364/364 ━━━━━━━━━━━━━━━━━━━━ 8s 21ms/step - accuracy: 0.8641 - loss: 0.4148 - val_accuracy: 0.7185 - val_loss: 0.9321


In [ ]:
# Evaluate the model

loss, accuracy = model.evaluate(X_test_pad, y_test)

print("Test Loss :", loss)
print("Test Accuracy :", accuracy)

114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7712 - loss: 0.7147
Test Loss : 0.7146944403648376
Test Accuracy : 0.7711771130561829


### Predictions of the BiLSTM model

In [ ]:
# Predict star ratings

y_pred = model.predict(X_test_pad)

# Convert probabilities into class labels

y_pred = np.argmax(y_pred, axis=1)

114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step


### Evaluation

In [ ]:
# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

# Append LSTM results to the results list

results.append({
    "Embedding": "Tokenizer + Embedding",
    "Model": "LSTM",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1})

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Classification Report

In [ ]:
# Classification Report

print(classification_report(y_test, y_pred))

# Confusion Matrix

print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.45      0.16      0.23       339
           1       0.00      0.00      0.00        56
           2       0.00      0.00      0.00        47
           3       0.22      0.10      0.14        98
           4       0.30      0.20      0.24       331
           5       0.82      0.97      0.89      2765

    accuracy                           0.77      3636
   macro avg       0.30      0.24      0.25      3636
weighted avg       0.70      0.77      0.72      3636

[[  54    0    0    9   26  250]
 [  15    0    0    9   16   16]
 [   5    0    0    9   19   14]
 [  15    0    0   10   33   40]
 [   5    0    0    4   65  257]
 [  27    0    0    5   58 2675]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Display Final Results

results_df = pd.DataFrame(results)
results_df

,Embedding,Model,Accuracy,Precision,Recall,F1 Score
0,Bag of Words,Logistic Regression,0.747800,0.701211,0.747800,0.719091
1,Bag of Words,SVM,0.708471,0.694571,0.708471,0.700810
2,TF-IDF,Logistic Regression,0.621837,0.736835,0.621837,0.663644
3,TF-IDF,SVM,0.778603,0.719110,0.778603,0.706423
4,Tokenizer + Embedding,LSTM,0.771177,0.700261,0.771177,0.723382


# Save the Model

In [ ]:

# Save the trained TF-IDF vectorizer for future text preprocessing

joblib.dump(tfidf, "tfidf_vectorizer.pkl")

# Save the trained SVM model for predicting recipe ratings

joblib.dump(svm, "recipe_rating_model.pkl")